In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "Computer_Science"
FEATURES = ["Maths", "Physics", "Chemistry", "English", "attendance_pct"]
RANDOM_STATE = 42


def load_data(filename):
    df = pd.read_csv(filename, dtype={"class": str})
    return df


def split_data(df):
    X = df[FEATURES]
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    return X_train, X_test, y_train, y_test


def evaluate_model(model, model_name, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    result = {
        "model_name": model_name,
        "predictions": predictions,
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "r2": round(r2, 3)
    }

    return result


def run_all_models(X_train, X_test, y_train, y_test):

    results = []

    model1 = LinearRegression()
    results.append(evaluate_model(model1, "Linear Regression", X_train, X_test, y_train, y_test))

    model2 = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE)
    results.append(evaluate_model(model2, "Decision Tree", X_train, X_test, y_train, y_test))

    model3 = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
    results.append(evaluate_model(model3, "Random Forest", X_train, X_test, y_train, y_test))

    model4 = RandomForestRegressor(
        n_estimators=200, max_depth=5, min_samples_leaf=3, random_state=RANDOM_STATE
    )
    results.append(evaluate_model(model4, "Random Forest - Tuned", X_train, X_test, y_train, y_test))

    return results


def get_baseline(y_train, y_test):
    mean_value = y_train.mean()
    predictions = np.full(len(y_test), mean_value)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    result = {
        "model_name": "Baseline (predict average)",
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "r2": round(r2, 3)
    }

    return result


def plot_results(results, y_test, filename):
    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    axes = axes.flatten()

    for i in range(len(results)):
        result = results[i]
        ax = axes[i]

        ax.scatter(y_test, result["predictions"], alpha=0.6)

        min_val = min(y_test.min(), result["predictions"].min())
        max_val = max(y_test.max(), result["predictions"].max())
        ax.plot([min_val, max_val], [min_val, max_val], "r--")

        ax.set_title(result["model_name"] + "\nR2 = " + str(result["r2"]))
        ax.set_xlabel("Actual Marks")
        ax.set_ylabel("Predicted Marks")

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


def find_best_model(results):
    best = results[0]

    for result in results:
        if result["r2"] > best["r2"]:
            best = result

    return best


def write_report(baseline, results, filename):

    file = open(filename, "w")

    file.write("PREDICTIVE MODELING REPORT\n")
    file.write("=" * 40 + "\n\n")

    file.write("Target: " + TARGET + " marks\n")
    file.write("Features: " + ", ".join(FEATURES) + "\n\n")

    file.write("MODEL COMPARISON\n")
    file.write("-" * 40 + "\n")
    file.write(baseline["model_name"] + " -> MAE: " + str(baseline["mae"]))
    file.write(", RMSE: " + str(baseline["rmse"]) + ", R2: " + str(baseline["r2"]) + "\n")

    for result in results:
        file.write(result["model_name"] + " -> MAE: " + str(result["mae"]))
        file.write(", RMSE: " + str(result["rmse"]) + ", R2: " + str(result["r2"]) + "\n")

    best_model = find_best_model(results)

    file.write("\nBEST MODEL\n")
    file.write("-" * 40 + "\n")
    file.write(best_model["model_name"] + " with R2 = " + str(best_model["r2"]) + "\n\n")

    file.write("INTERPRETATION\n")
    file.write("-" * 40 + "\n")

    if best_model["r2"] > baseline["r2"] + 0.05 and best_model["r2"] > 0.3:
        file.write("The best model explains a good amount of the variation in marks\n")
        file.write("and clearly beats the baseline. It has learned a real pattern.\n")
    elif best_model["r2"] > baseline["r2"] + 0.05:
        file.write("The best model beats the baseline a little. There is some signal\n")
        file.write("but predictions should be treated as rough, not exact.\n")
    else:
        file.write("No model beat the baseline in any meaningful way. This means a\n")
        file.write("student's marks in one subject do not really predict their marks\n")
        file.write("in another subject in this dataset. That is still a useful and\n")
        file.write("honest finding to report.\n")

    file.close()


def main():

    df = load_data("clean_student_marks.csv")
    X_train, X_test, y_train, y_test = split_data(df)

    baseline = get_baseline(y_train, y_test)
    results = run_all_models(X_train, X_test, y_train, y_test)

    plot_results(results, y_test, "model_predictions.png")
    write_report(baseline, results, "model_comparison_report.txt")

    print("Modeling done!")
    print("Baseline R2:", baseline["r2"])
    for result in results:
        print(result["model_name"], "R2:", result["r2"], "MAE:", result["mae"])
    print("Files saved: model_comparison_report.txt, model_predictions.png")


if __name__ == "__main__":
    main()

Modeling done!
Baseline R2: -0.004
Linear Regression R2: -0.173 MAE: 18.81
Decision Tree R2: 0.009 MAE: 17.5
Random Forest R2: -0.135 MAE: 17.88
Random Forest - Tuned R2: -0.115 MAE: 17.85
Files saved: model_comparison_report.txt, model_predictions.png
